# Day 1 · 01 · Lakehouse, Delta Tables, SQL Warehouse & Analyst Tooling

![Medallion to Power BI](../../assets/images/medallion_to_powerbi.png)

This demo introduces the platform from an analyst perspective. Databricks is not only a SQL endpoint: it combines governed open storage, Delta transaction guarantees, SQL compute and tools that help analysts discover data before modeling it.

## Program Map for Power BI Developers

This course is organized around the decisions a Power BI developer must make when Databricks becomes the governed data platform.

| Session | Module | Status | Goal for Power BI developers | End artifact |
|---|---|---|---|---|
| Session 1 | SQL Warehouse and tooling | Required | understand where Power BI queries run and how to inspect trusted data | source discovery notes and warehouse guardrails |
| Session 1 | Gold and KPI best practices | Required | learn BI-safe SQL patterns, grain and KPI validation | modeling rules for Workshop 1 |
| Session 1 | Workshop 1 Gold model | Required | build the governed Gold contract consumed later by Power BI | `gold.dim_*`, `gold.fact_sales`, `gold.fact_sales_dashboard` |
| Session 1 | W1 Gold handoff checkpoint | Optional | confirm the Gold model is ready for Day 2 | compact inventory and KPI smoke check |
| Session 2 | **Power BI connection & Delta Sharing** | **Required** | **live demo: connect PBI Desktop to SQL Warehouse, Import vs DirectQuery, Delta Sharing** | **connection notes, .share file** |
| Session 2 | Power BI semantic model | Required | decide star vs wide, measures, relationships and refresh mode | Power BI handoff packet |
| Session 2 | Workshop 2 dataset performance | Required | build serving objects for Import, drill-through and incremental refresh | monthly aggregate, incremental view, KPI table |
| Session 2 | Performance and automation orientation | Required | troubleshoot slow reports and understand production refresh ownership | BI performance and operations checklist |
| Session 2 | AI/BI Dashboards and Genie | Optional | understand Databricks-native BI as a complement to Power BI | dashboard and Genie checklist |

Expected observation: the course is not a data-engineering certification path. It teaches how Power BI developers consume, validate and troubleshoot Databricks-backed data products.

## Business Scenario

RetailHub has multiple Power BI reports that disagree on sales KPIs. Before building a Gold model, the team needs a shared understanding of where trusted data lives, how SQL Warehouses serve BI, and how to inspect Delta tables safely.

## Power BI Developer Lens

As a Power BI developer, treat Databricks as the upstream platform that decides whether your reports are trusted, fast and governable.

| Databricks concept | Power BI meaning | Question to ask |
|---|---|---|
| SQL Warehouse | query backend for SQL Editor, Databricks SQL and Power BI connector | which warehouse should my report use? |
| Unity Catalog | governed catalog of tables, permissions, lineage and ownership | is this table certified and who owns it? |
| Delta table | stable, versioned table with history | did the data change when my KPI changed? |
| Gold schema | approved BI contract | is this the source I should model in Power BI? |
| Query History/Profile | evidence for slow DirectQuery reports | which visual/query is expensive and why? |

Trainer note: keep returning to these questions. The technical feature matters only when it changes model design, refresh, DirectQuery cost or KPI trust.

## Learning Objectives

By the end of this notebook you can:

- explain Lakehouse vs classic Data Warehouse in BI terms,
- describe Delta table internals as Parquet files plus a transaction log,
- inspect Delta metadata with `DESCRIBE DETAIL` and `DESCRIBE HISTORY`,
- explain SQL Warehouse types, cost drivers and guardrails,
- use SQL Editor, Catalog Explorer and notebooks for different analyst tasks,
- run a source discovery pattern from `sql/exploration_queries.sql`,
- identify row counts, grain, status distribution and date range before modeling.

Documentation starting points: [Databricks Lakehouse](https://docs.databricks.com/lakehouse/), [Delta Lake](https://docs.databricks.com/delta/), [SQL Warehouses](https://docs.databricks.com/compute/sql-warehouse/).

## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `silver.order_lines`.

In [0]:
%run ../../setup/00_setup

### Configuration

Confirm the active catalog and schemas.

In [0]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

This notebook requires the four Silver source tables created by `data/generate_training_dataset.ipynb`.

In [0]:
required_tables = [
    f"{SILVER}.customers",
    f"{SILVER}.products",
    f"{SILVER}.sales_orders",
    f"{SILVER}.order_lines",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run data/generate_training_dataset.ipynb first.")
print("[OK] Silver source tables are available.")

## 1. Lakehouse vs Data Warehouse

![Lakehouse vs Data Warehouse](../../assets/images/day1_01_lakehouse_vs_data_warehouse.png)

Both architectures can serve BI, but the Lakehouse changes where trust and scale come from. Data stays in open cloud storage, Delta Lake adds transactional guarantees, Unity Catalog governs access and lineage, and SQL Warehouse serves analytical queries.

Hands-on step: run the next Catalog Explorer SQL cells and confirm that the training catalog exposes governed schemas instead of isolated report extracts.

Expected observation: participants should see a catalog with Bronze, Silver and Gold schemas, then use Silver for discovery and Gold for BI-ready consumption.

### RetailHub Source Map

![RetailHub source map](../../assets/images/retailhub_source_map.png)

Use this picture as the first business anchor. The source system exposes customer, product, order header and order line data. Day1_01 only discovers these sources; Workshop 1 turns them into a governed Gold model.

Trainer prompt: ask participants which table is likely at order grain and which table is likely at line grain before running any SQL.

### Unity Catalog Context

Unity Catalog is the governance layer: catalogs, schemas, tables, comments, lineage, ownership and permissions. Catalog Explorer is where analysts inspect this contract before connecting a BI tool.

Power BI developer lens: before using a table in a report, check ownership, comments and lineage. If there is no clear owner or description, treat it as an exploration source, not a certified report source.


### Catalog Explorer: Hierarchy

![Catalog Explorer hierarchy](../../assets/images/source_catalog_explorer_hierarchy.png)

Catalog Explorer is the analyst's map of governed data. The hierarchy is `catalog -> schema -> table/view -> columns`. In this course, the participant catalog contains `bronze`, `silver` and `gold` schemas.

Expected observation: before modeling, analysts should check table ownership, comments, columns and sample data instead of guessing from names alone.

In [0]:
%sql
SHOW CATALOGS

### Schemas in the Participant Catalog

The training catalog uses Bronze, Silver and Gold to express Medallion layers.

In [0]:
%sql
SHOW SCHEMAS

### Catalog Explorer: Lineage

![Catalog Explorer lineage](../../assets/images/source_catalog_explorer_lineage.png)

Lineage answers a different question than schema browsing: **where did this object come from and what depends on it?** For BI teams, lineage helps assess the blast radius of changing a Gold table, renaming a column or changing a KPI definition.

## 2. Delta Table = Parquet Files + Transaction Log

![Delta table anatomy](../../assets/images/day1_01_delta_table_anatomy.png)

A Delta table is not just a folder of files. It combines Parquet data files with the `_delta_log` transaction log, so SQL readers get a consistent table snapshot and an auditable history of changes.

Hands-on step: inspect `silver.order_lines` with `DESCRIBE DETAIL` and `DESCRIBE HISTORY`.

Expected observation: `format` should be `delta`, the table should have active data files, and history should show at least one committed operation.

Power BI developer lens: Delta history helps explain KPI changes. If a report changed after refresh, `DESCRIBE HISTORY` is the first evidence point before blaming DAX.


### Delta Function Glossary

`DESCRIBE DETAIL` returns physical and logical table metadata: format, location, file count, size and properties. Use it to prove that a table is Delta and to estimate how much data a SQL query may scan.

`DESCRIBE HISTORY` reads the Delta transaction log and shows committed operations such as create, write, update, optimize and restore. Use it when a KPI changed and you need to know whether the table changed.

`VERSION AS OF` pins a query to an earlier Delta snapshot. It is read-only and useful for investigation; it is not a replacement for a governed rollback process.

Pitfall: time travel depends on retained history and data files. Aggressive `VACUUM` can remove files needed by older snapshots.


### Table Detail

`DESCRIBE DETAIL` shows the table format, file count, size, location and table properties. The location points to the directory that contains data files and `_delta_log`.

In [0]:
%sql
DESCRIBE DETAIL silver.order_lines

### Operation History

`DESCRIBE HISTORY` is the audit trail for Delta operations. It helps answer what changed, when it changed and which operation performed the change.

In [0]:
%sql
DESCRIBE HISTORY silver.order_lines LIMIT 10

### Time Travel: Read an Earlier Snapshot

Time travel is a Delta feature that lets you query a previous committed table version. For analysts it is useful when a KPI changed and you need to compare today's result with a known earlier snapshot.

Hands-on step: compare the current row count with version `0` of the same Delta table.

Expected observation: both reads use the same table name, but `VERSION AS OF` pins the query to an older transaction-log snapshot.


In [0]:
%sql
SELECT
  'current' AS snapshot,
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date
FROM silver.order_lines
UNION ALL
SELECT
  'version_0',
  COUNT(*),
  COUNT(DISTINCT line_id),
  MIN(order_date),
  MAX(order_date)
FROM silver.order_lines VERSION AS OF 0


In [0]:
%sql
SELECT
  line_id,
  order_id,
  order_date,
  status,
  line_revenue,
  line_margin
FROM silver.order_lines VERSION AS OF 0
ORDER BY line_id
LIMIT 10


### Schema and Comments

`DESCRIBE EXTENDED` exposes column names, data types and metadata. For BI work, comments and stable column names are part of the data contract.


### Schema Enforcement and Schema Evolution

Delta Lake validates every write against the table schema. Both features protect data quality and pipeline stability.

**Schema Enforcement — Delta rejects incompatible writes**

When writing to a Delta table, Databricks checks that incoming data matches the table schema. Writes with wrong column types are rejected before a single row is committed — no partial writes.

```sql
-- FAILS if registration_date column is DATE but incoming value is STRING
-- Delta rejects the entire write atomically
INSERT INTO silver.customers (customer_id, registration_date)
VALUES ('CUST999', '2024-01-01');   -- '2024-01-01' as string → rejected
```

**Schema Evolution — Controlled addition of new columns**

Schema evolution allows adding new columns to an existing Delta table without breaking existing pipelines. Old rows receive `NULL` in the new column.

```python
# Opt-in to schema evolution in PySpark
df_with_new_column.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.customers")
```

```sql
-- Or add a column explicitly in SQL
ALTER TABLE silver.customers ADD COLUMN loyalty_tier STRING;
```

| Feature | Schema Enforcement | Schema Evolution |
|---------|-------------------|-----------------|
| What it does | Rejects writes that don't match the schema | Allows adding new columns to existing tables |
| Opt-in required | No — always active by default | Yes — `mergeSchema = True` or `ALTER TABLE ADD COLUMN` |
| Impact on existing rows | Protects them from unexpected changes | New column is `NULL` for existing rows |
| Impact on existing pipelines | Protected from schema drift | No breakage — additive only |

In [0]:
%sql
DESCRIBE EXTENDED silver.sales_orders


## 3. Delta Deep Dive: RESTORE, OPTIMIZE and VACUUM

`RESTORE`, `OPTIMIZE` and `VACUUM` are operational Delta commands. We do not run them on source Silver tables in a Day 1 analyst demo. Instead, we create a tiny disposable Delta table in Gold, mutate it, restore it and then clean it up.

Expected observation: Delta history records every table operation. `RESTORE` adds another transaction; it does not manually edit Parquet files.


In [0]:
%sql
CREATE OR REPLACE TABLE gold.__day1_delta_restore_demo
COMMENT 'Disposable Day 1 demo table for Delta RESTORE and OPTIMIZE awareness.'
AS
SELECT
  line_id,
  order_id,
  order_date,
  status,
  line_revenue,
  line_margin
FROM silver.order_lines
ORDER BY line_id
LIMIT 20


In [0]:
%sql
SELECT
  'after_create' AS checkpoint,
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines
FROM gold.__day1_delta_restore_demo


### Create a New Delta Version

The next statement intentionally changes one row in the disposable table. This creates a new Delta version that we can restore from.


In [0]:
%sql
UPDATE gold.__day1_delta_restore_demo
SET status = 'Demo Changed'
WHERE line_id = (
  SELECT MIN(line_id)
  FROM gold.__day1_delta_restore_demo
)


In [0]:
%sql
SELECT
  status,
  COUNT(*) AS rows
FROM gold.__day1_delta_restore_demo
GROUP BY status
ORDER BY rows DESC, status


In [0]:
%sql
DESCRIBE HISTORY gold.__day1_delta_restore_demo


### RESTORE Definition and Safety Rule

`RESTORE` creates a new Delta transaction that makes the table state match an earlier version. It does not delete the audit trail; history still shows the restore operation.

Use `RESTORE` for controlled rollback after a bad load or accidental update. Do not run it casually on shared Silver or Gold tables during analyst exploration.


### RESTORE the Disposable Table

`RESTORE TABLE ... TO VERSION AS OF ...` rolls the Delta table back to a selected version. It is powerful and should be used with intent, because downstream readers will see the restored state.

Hands-on step: restore the disposable table to version `0` and check that the demo update disappeared.


In [0]:
%sql
RESTORE TABLE gold.__day1_delta_restore_demo TO VERSION AS OF 0


In [0]:
%sql
SELECT
  status,
  COUNT(*) AS rows
FROM gold.__day1_delta_restore_demo
GROUP BY status
ORDER BY rows DESC, status


### OPTIMIZE and VACUUM Definitions

`OPTIMIZE` compacts many small Parquet files into fewer larger files. It can reduce scan overhead for SQL Warehouse and BI queries, especially after many small writes.

`VACUUM` removes obsolete data files that are no longer referenced by the Delta log retention policy. It saves storage, but it can also shorten the time travel window.

Pitfall: `VACUUM ... DRY RUN` is safe for training because it lists candidates without deleting files.


### OPTIMIZE and VACUUM Awareness

`OPTIMIZE` compacts small files so SQL Warehouse reads fewer files. `VACUUM` removes old data files that are no longer needed by the Delta log retention policy, which can reduce the time travel window.

For Day 1, the important analyst takeaway is cost and performance awareness: lots of tiny files can make BI queries slower and more expensive.


In [0]:
%sql
OPTIMIZE gold.__day1_delta_restore_demo


In [0]:
%sql
VACUUM gold.__day1_delta_restore_demo RETAIN 168 HOURS DRY RUN


In [0]:
%sql
DROP TABLE IF EXISTS gold.__day1_delta_restore_demo


### Identity and Generated Columns

Delta tables support two types of auto-computed columns — useful for surrogate keys and derived fields in Gold models.

| Column type | Syntax | Behavior |
|-------------|--------|----------|
| `IDENTITY` | `col BIGINT GENERATED ALWAYS AS IDENTITY` | Auto-incrementing surrogate key — unique and increasing, not necessarily contiguous |
| Generated | `col TYPE GENERATED ALWAYS AS (expression)` | Computed from other columns at write time — cannot be set manually |
| `IDENTITY BY DEFAULT` | `col BIGINT GENERATED BY DEFAULT AS IDENTITY` | Like `ALWAYS`, but allows explicit values to be inserted |

```sql
CREATE TABLE gold.fact_sales_sk (
    sale_sk     BIGINT GENERATED ALWAYS AS IDENTITY,
    order_id    STRING,
    sale_date   DATE,
    year_month  STRING GENERATED ALWAYS AS (DATE_FORMAT(sale_date, 'yyyy-MM'))
) USING DELTA;
```

> **Analyst note:** IDENTITY columns are not guaranteed to be contiguous — gaps are normal in distributed systems. Use `sale_sk` for joins and relationships, not as a sequence counter.

### Managed vs External Tables

Unity Catalog distinguishes two table types. The critical difference is what happens when you run `DROP TABLE`.

| Feature | Managed Table | External Table |
|---------|--------------|----------------|
| Data location | Databricks controls the file path | You specify `LOCATION 'abfss://...'` |
| `DROP TABLE` effect | Removes the table **and deletes all data files** | Removes only the catalog entry — **files remain in storage** |
| Lifecycle | Managed by Databricks | Managed by you |
| When to use | Most cases — simpler, recommended default | Data shared with external systems; migration; data that must outlive the table definition |

```sql
-- Managed table (default — no LOCATION)
CREATE TABLE gold.fact_sales (...) USING DELTA;

-- External table — files survive DROP TABLE
CREATE TABLE gold.fact_sales_ext (...)
USING DELTA
LOCATION 'abfss://container@storage.dfs.core.windows.net/gold/fact_sales';

-- Check table type
SELECT table_name, table_type
FROM information_schema.tables
WHERE table_schema = 'gold';
```

> **Key rule:** Use Managed Tables by default. Use External only when data must survive the table lifecycle in Databricks — for example when it is shared with external ETL tools or other catalog systems.

### Common Delta Errors and Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `ConcurrentAppendException` | Two writers appended to the same partition simultaneously | Avoid parallel appends to the same partition; use `SERIALIZABLE` isolation |
| `A schema mismatch found when writing to a Delta table` | Column names or types don't match table schema | Check with `DESCRIBE TABLE` — enable `mergeSchema` only if evolution is intentional |
| `MERGE found multiple matches` | One source row matches multiple target rows on the join key | Ensure the ON condition uses a unique key; deduplicate source before MERGE |
| `Version N is not available` | Requested Time Travel version was vacuumed away | Increase `delta.logRetentionDuration` or use a version within the retention window |
| `DELTA_MISSING_TRANSACTION_LOG` | `_delta_log/` directory deleted manually | Never delete `_delta_log/` directly — use `DROP TABLE` instead |

> **Quick diagnostic:** `DESCRIBE HISTORY <table>` shows recent operations and helps identify what changed before the error occurred.

## 4. SQL Warehouse: Definition, Types and Cost Model

![SQL Warehouse cost decision](../../assets/images/sql_warehouse_cost_decision.png)

A **SQL Warehouse** is Databricks compute optimized for SQL workloads. It is the endpoint used by Databricks SQL, SQL Editor, dashboards, alerts and BI tools such as Power BI. The warehouse executes queries; data remains in governed Delta tables in Unity Catalog.

For an analyst, the shortest definition is: **SQL Warehouse is the BI query engine for Lakehouse tables**.

### What a SQL Warehouse Is and Is Not

A SQL Warehouse is Databricks compute for SQL analytics and BI connectivity. It runs SQL statements against governed Unity Catalog objects and is the endpoint Power BI uses through the Databricks connector.

It is not a storage layer, not a replacement for Unity Catalog permissions and not the place where report-specific DAX logic should live. It can execute SQL that creates objects, but the modeling decision still belongs to the analytics workflow.

Hands-on step: confirm the active SQL context in the notebook, then compare it with the SQL Warehouse connection details shown in the workspace UI.

Expected observation: Power BI handoff needs SQL Warehouse connection details even if this notebook is running on all-purpose or serverless notebook compute.

### Where It Fits in the BI Architecture

![SQL Warehouse architecture](../../assets/images/day1_01_sql_warehouse_architecture.png)

SQL Editor, Databricks dashboards and Power BI send SQL to a SQL Warehouse. The warehouse enforces governance through Unity Catalog and reads Delta tables or views from Silver and Gold.

Hands-on step: after running the active warehouse context query, open the SQL Warehouse UI and locate the same warehouse under **SQL Warehouses**.

Expected observation: stopping or resizing the warehouse affects query compute, not the Delta tables stored in the catalog.

### Compute Decision Map

![Compute decision map](../../assets/images/day1_01_compute_decision_map.png)

Choose compute from the workload, not from habit. SQL analytics and BI tools use SQL Warehouse; Python/PySpark exploration uses notebook compute; scheduled repeatable work uses job compute.

For SQL Warehouses, the practical Day 1 decision is simple: use Serverless when it is available and permitted; otherwise use Pro. Treat Classic as a compatibility option.

Expected observation: a BI report should not depend on an interactive notebook cluster.

### Compute Types at a Glance

| Compute type | Typical use case | Startup time | Cost model |
|---|---|---|---|
| **All-Purpose Cluster** | Interactive notebooks, data exploration | 2–5 min | Per DBU/hour while running |
| **Job Cluster** | Scheduled pipeline runs | 2–4 min | Per DBU/hour, auto-terminates after job |
| **SQL Warehouse** | BI queries, SQL Editor, dashboards | < 5 s (Serverless) | Per DBU/hour while active queries run |
| **Serverless Notebook** | On-demand notebook compute | < 10 s | Per DBU/second of actual usage |

**Rule of thumb:** For ad-hoc analyst SQL and Power BI — SQL Warehouse. For scheduled Delta pipelines — Job Cluster. For interactive data engineering exploration — All-Purpose Cluster.

### Warehouse Settings That Control User Experience

The most important settings for analysts are warehouse type, size, min/max clusters and auto-stop. Together they control startup behavior, concurrency, latency and idle cost.

Hands-on UI step: open **SQL Warehouses -> your warehouse -> Edit** and identify the warehouse type, size, scaling range and auto-stop value.

Expected observation: sizing is not a one-time guess. Start small, observe Query History and then scale based on evidence.

### SQL Warehouse Operational Checklist

A SQL Warehouse is compute, so operational settings directly affect cost and user experience.

| Setting | Definition | Practical rule |
|---|---|---|
| Auto-stop | stops idle compute after a configured time | keep it short for training and ad hoc BI |
| Warehouse size | controls compute capacity per cluster | start small, resize after evidence from Query History |
| Max clusters | controls concurrency scaling | cap it for cost guardrails |
| Query timeout | stops long-running queries | prevents runaway DirectQuery or broad scans |
| Permissions | controls who can use/start/manage warehouse | report consumers usually need use, not manage |
| Naming convention | makes ownership and purpose visible | include environment, team and workload |

Expected observation: warehouse governance is not only permissions. It is also sizing, idle behavior and concurrency limits.


### Cost Model: What Actually Drives Spend

![BI warehouse cost model](../../assets/images/day1_01_cost_model.png)

Do not hard-code prices in training materials. Pricing changes by cloud, region, SKU, contract and workspace features, but the operating model is stable: active time, warehouse size, running clusters and query fan-out drive cost pressure.

Hands-on step: run the query fan-out mini model below and change the parameter values during delivery.

Expected observation: DirectQuery-style interaction volume can multiply SQL queries even when the report has only a few visuals.

### Import vs DirectQuery Cost Intuition

![Import vs DirectQuery](../../assets/images/import_vs_directquery.png)

Import usually pays the warehouse cost during scheduled refresh. DirectQuery can pay on page load, filter changes and user interactions. This is why DirectQuery belongs on carefully designed Gold objects, not raw line-grain Silver tables.

Hands-on step: use the next mini model to estimate how visuals, users and clicks become SQL query volume.

Expected observation: the best cost control is often a better Gold source, not only a smaller warehouse.

### Query Fan-out Mini Model

Change the numbers in the `params` CTE during delivery. This is not a billing calculator; it is a mental model for why DirectQuery governance matters.

### SQL Warehouse Cost Scenarios

Use these scenarios to explain why the same dataset can have different cost behavior depending on the Power BI mode.


In [0]:
%sql
WITH scenarios AS (
  SELECT 'Import refresh' AS scenario, 1 AS report_opens, 1 AS refreshes, 6 AS visuals, 1 AS interactions_per_open
  UNION ALL SELECT 'DirectQuery dashboard', 20, 0, 6, 5
  UNION ALL SELECT 'Ad hoc analyst query', 1, 0, 1, 8
)
SELECT
  scenario,
  report_opens,
  refreshes,
  visuals,
  interactions_per_open,
  (refreshes * visuals) + (report_opens * visuals * interactions_per_open) AS estimated_query_events
FROM scenarios
ORDER BY estimated_query_events DESC


Expected observation: DirectQuery cost is driven by user interactions and visual fan-out. Import cost is usually concentrated in refresh windows.


In [0]:
%sql
WITH params AS (
  SELECT
    8 AS visuals_on_page,
    12 AS interactions_per_user,
    25 AS concurrent_users,
    2 AS avg_queries_per_visual
)
SELECT
  visuals_on_page,
  interactions_per_user,
  concurrent_users,
  avg_queries_per_visual,
  visuals_on_page * interactions_per_user * concurrent_users * avg_queries_per_visual AS estimated_sql_queries
FROM params

### Active Warehouse Context

Expected observation: the notebook confirms user, catalog and schema. SQL Warehouse hostname and HTTP path are taken from the SQL Warehouse connection details UI, not from this notebook session.

In [0]:
%sql
SELECT
  current_user() AS current_user,
  current_catalog() AS catalog,
  current_schema() AS schema

### Query History and Query Profile: Where Evidence Comes From

`Query History` is the audit and troubleshooting view for SQL execution. It shows who ran a query, when it ran, which warehouse served it, how long it took and whether it failed.

`Query Profile` is the execution breakdown for a single query. It helps identify whether time was spent scanning files, filtering rows, joining, shuffling or aggregating.

How to read it in Day 1 terms:

- Scan-heavy query: many bytes read before filters reduce rows; consider better Gold shape or clustering/optimization awareness.
- Filter not selective: the `WHERE` clause exists but still reads most data; check columns and date filters.
- DirectQuery fan-out: many similar queries from one report page; reduce visuals, use Import or create a narrower Gold table.
- Long aggregation: repeated group-bys over line-grain tables; consider pre-aggregated Gold only when the business grain is stable.

Expected observation: professional warehouse sizing should be based on Query History and Query Profile evidence, not guesswork.

Power BI developer lens: Performance Analyzer tells you which visual is slow; Query History and Query Profile tell you what Databricks actually executed for that visual or refresh.


### UI Walkthrough: Where to Click

Trainer-led workspace demo:

1. Open **SQL Warehouses** from the left navigation.
2. Select the training warehouse.
3. On **Overview**, show state, size and running clusters.
4. On **Edit**, show type, cluster size, scaling and auto-stop.
5. On **Connection details**, show **Server hostname** and **HTTP path** for Power BI.
6. On **Query History / Monitoring**, show the latest queries from this notebook or SQL Editor.

This click path matters because participants will need it again during Power BI handoff.

### Cost Guardrails for This Course

Use these guardrails during the hands-on demo:

- Start with a small warehouse and scale only from Query History evidence.
- Keep auto-stop short for training and development, usually 5-10 minutes.
- Use Import by default for executive BI pages; reserve DirectQuery for real live-data requirements.
- Serve Power BI from Gold tables or aggregates, not wide unfiltered detail sources.
- Separate BI, jobs and ad-hoc exploration if workloads interfere with each other.

Expected observation: cost control is a design discipline, not only an administrator setting.

### Reference Links

Use these as live references because product capabilities and pricing change:

- [SQL warehouse types](https://docs.databricks.com/aws/en/compute/sql-warehouse/warehouse-types)
- [Create and configure a SQL warehouse](https://docs.databricks.com/aws/en/compute/sql-warehouse/create)
- [SQL warehouse sizing, scaling and queuing behavior](https://docs.databricks.com/aws/en/compute/sql-warehouse/warehouse-behavior)
- [Cost optimization best practices](https://docs.databricks.com/aws/en/lakehouse-architecture/cost-optimization/best-practices)
- [Databricks pricing](https://www.databricks.com/product/pricing)

## 5. Analyst Tooling

Analyst work moves through a lifecycle: fast exploration, repeatable checks and governed consumption. SQL Editor, Catalog Explorer, notebooks and Power BI each fit a different part of that lifecycle.

Hands-on rule: use SQL Editor for quick exploration, notebooks for repeatable training checks and Gold objects for BI consumption.

Expected observation: the same SQL pattern can appear in SQL Editor and in a notebook, but ownership and lifecycle are different.

### Magic Commands Quick Reference

| Magic | Language / Action | Typical use |
|---|---|---|
| `%sql` | Switch cell to SQL | Run SQL in a Python notebook |
| `%python` | Switch cell to Python | Run Python in a SQL notebook |
| `%md` | Markdown | Section headers, documentation |
| `%fs` | DBFS filesystem (`ls`, `cp`, `rm`) | Browse or manage files |
| `%sh` | Shell commands | Install system packages, check OS |
| `%run` | Execute another notebook | Load shared setup, run child notebook |
| `%pip install` | Install Python package in session | Add libraries without cluster restart |

**Keyboard shortcuts:** `Ctrl+/` (Win) or `Cmd+/` (Mac) — toggle comment on selected lines.

### Databricks AI Assistant

Open with **`Cmd+I`** (Mac) or **`Ctrl+I`** (Win) in any notebook cell.

| Use case | Example prompt |
|---|---|
| Generate SQL | "Write a SQL query that counts orders per channel for the last 30 days" |
| Explain code | "Explain what this window function does" |
| Diagnose errors | Paste the error message and ask "why does this fail?" |
| Refactor | "Rewrite this subquery as a CTE" |

The assistant has context of the current cell and recent notebook output. Use it for first-draft SQL, quick explanations and error diagnosis — always review the result before running.

### Tooling Workflow Visual

![Databricks SQL Editor](../../assets/images/source_sql_editor.png)

A professional workflow usually moves through three states: quick exploration in SQL Editor, repeatable checks in notebooks, and governed consumption from Gold tables in Power BI. The same SQL pattern can exist in all three places, but ownership and lifecycle are different.

### SQL Editor Exercise: External Query File

Open `sql/exploration_queries.sql` and run it in Databricks SQL Editor. This is a real saved-query style artifact, not a notebook-only example.

Hands-on steps:

1. Open **SQL Editor** and create a new query.
2. Paste the contents of `sql/exploration_queries.sql`.
3. Replace `YOUR_CATALOG` with your participant catalog.
4. Select the training SQL Warehouse.
5. Create these SQL Editor parameters:
   - `p_status` = `Completed`
   - `p_start_date` = `2024-01-01`
   - `p_end_date` = `2024-12-31`
   - `p_channel` = `All`
   - `p_top_n` = `10`
6. Run one section at a time and change parameter values during the demo.

Expected observation: SQL Editor parameters use `:parameter_name`, for example `WHERE status = :p_status`. The same exploration logic is reproduced below in notebook `%sql` cells for repeatable training checks.

### Row Counts Across Silver

This reproduces section `02. Row counts across Silver tables` from `sql/exploration_queries.sql`.

Expected observation: row counts establish scale before participants interpret any KPI.

In [0]:
%sql
SELECT 'customers' AS table_name, COUNT(*) AS rows FROM silver.customers
UNION ALL
SELECT 'products', COUNT(*) FROM silver.products
UNION ALL
SELECT 'sales_orders', COUNT(*) FROM silver.sales_orders
UNION ALL
SELECT 'order_lines', COUNT(*) FROM silver.order_lines

### Parameterized Query Pattern

SQL Editor uses named parameters such as `:p_status`, `:p_start_date` and `:p_channel`. In this notebook we keep the same intent visible with a one-row `params` CTE, because notebook SQL cells do not need the SQL Editor parameter widget to demonstrate the pattern.

Expected observation: the filter logic is the same even though SQL Editor and notebook use different parameter mechanics.

In [0]:
%sql
WITH params AS (
  SELECT 'Completed' AS p_status
)
SELECT
  o.status,
  o.channel,
  COUNT(DISTINCT o.order_id) AS orders
FROM silver.sales_orders o
CROSS JOIN params p
WHERE o.status = p.p_status
GROUP BY o.status, o.channel
ORDER BY orders DESC

### Date Window Parameter Pattern

This reproduces the SQL Editor date/channel pattern with a notebook-local `params` CTE. Use it to show that parameters are not a gimmick: they make the filter contract visible.

Expected observation: changing `p_status`, `p_start_date`, `p_end_date` or `p_channel` in the CTE changes the result without rewriting the business query.

In [0]:
%sql
WITH params AS (
  SELECT
    'Completed' AS p_status,
    DATE '2024-01-01' AS p_start_date,
    DATE '2024-12-31' AS p_end_date,
    'All' AS p_channel
)
SELECT
  l.channel,
  COUNT(DISTINCT l.order_id) AS orders,
  ROUND(SUM(l.line_revenue), 2) AS revenue,
  ROUND(SUM(l.line_margin), 2) AS gross_margin,
  ROUND(100.0 * SUM(l.line_margin) / NULLIF(SUM(l.line_revenue), 0), 2) AS margin_rate_pct
FROM silver.order_lines l
CROSS JOIN params p
WHERE l.status = p.p_status
  AND l.order_date BETWEEN p.p_start_date AND p.p_end_date
  AND (p.p_channel = 'All' OR l.channel = p.p_channel)
GROUP BY l.channel
ORDER BY revenue DESC

### Top-N Parameter Pattern

This reproduces the `p_top_n` example from the SQL Editor script. Top-N queries are common in stakeholder demos, but the limit should be explicit and easy to change.

Expected observation: the output rank cutoff comes from the `params` CTE, not from a hidden magic number in the final `WHERE` clause.

In [0]:
%sql
WITH params AS (
  SELECT
    'Completed' AS p_status,
    DATE '2024-01-01' AS p_start_date,
    DATE '2024-12-31' AS p_end_date,
    'All' AS p_channel,
    10 AS p_top_n
),
product_revenue AS (
  SELECT
    l.product_id,
    l.category,
    ROUND(SUM(l.line_revenue), 2) AS revenue
  FROM silver.order_lines l
  CROSS JOIN params p
  WHERE l.status = p.p_status
    AND l.order_date BETWEEN p.p_start_date AND p.p_end_date
    AND (p.p_channel = 'All' OR l.channel = p.p_channel)
  GROUP BY l.product_id, l.category
),
ranked AS (
  SELECT
    product_id,
    category,
    revenue,
    DENSE_RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
  FROM product_revenue
)
SELECT
  revenue_rank,
  product_id,
  category,
  revenue
FROM ranked
CROSS JOIN params p
WHERE revenue_rank <= p.p_top_n
ORDER BY revenue_rank, product_id

## 6. Source Discovery Before Gold

![Source risk register](../../assets/images/day1_01_source_risk_register.png)

Before building any model, prove the grain and business states. Most BI defects start with an incorrect assumption about what one row means or which rows count toward a KPI.

Hands-on step: run the discovery queries below and translate the results into modeling risks.

Expected observation: Day1_01 discovers risk; Workshop 1 decides how those risks become Gold modeling rules.

### Order Header Grain and Date Range

In [0]:
%sql
SELECT
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date,
  COUNT(*) AS rows,
  COUNT(DISTINCT order_id) AS distinct_orders
FROM silver.sales_orders

### Status Distribution

In [0]:
%sql
SELECT
  status,
  COUNT(*) AS orders
FROM silver.sales_orders
GROUP BY status
ORDER BY orders DESC

### Order Line Grain

In [0]:
%sql
SELECT
  COUNT(*) AS line_rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(DISTINCT order_id) AS distinct_orders,
  ROUND(COUNT(*) / COUNT(DISTINCT order_id), 2) AS avg_lines_per_order
FROM silver.order_lines

### Source Risk Register Query

This query turns discovery into a risk register. It does not fix the data; it tells the team which modeling decisions must be made before Power BI consumes Gold.

Expected observation: at least some risks are intentionally present in the training dataset, so participants have real issues to discuss in Workshop 1.

In [0]:
%sql
SELECT 'unknown_status' AS risk_name, COUNT(*) AS affected_rows
FROM silver.sales_orders
WHERE status NOT IN ('Completed', 'Returned', 'Cancelled')
UNION ALL
SELECT 'future_dated_orders', COUNT(*)
FROM silver.sales_orders
WHERE order_date > current_date()
UNION ALL
SELECT 'null_unit_price_lines', COUNT(*)
FROM silver.order_lines
WHERE unit_price IS NULL
UNION ALL
SELECT 'orphan_customer_lines', COUNT(*)
FROM silver.order_lines l
LEFT ANTI JOIN silver.customers c ON l.customer_id = c.customer_id
UNION ALL
SELECT 'orphan_product_lines', COUNT(*)
FROM silver.order_lines l
LEFT ANTI JOIN silver.products p ON l.product_id = p.product_id
ORDER BY affected_rows DESC

### Header-vs-Lines Reconciliation Sample

Order headers and order lines often come from different source-system paths. Reconciliation checks whether the header total agrees with the line-level total.

Expected observation: mismatches are modeling and data-quality evidence, not something to hide in a visual.

In [0]:
%sql
WITH line_totals AS (
  SELECT
    order_id,
    ROUND(SUM(line_revenue), 2) AS line_revenue_total
  FROM silver.order_lines
  GROUP BY order_id
)
SELECT
  o.order_id,
  o.order_date,
  o.status,
  o.order_total_amount,
  l.line_revenue_total,
  ROUND(o.order_total_amount - l.line_revenue_total, 2) AS amount_diff
FROM silver.sales_orders o
JOIN line_totals l ON o.order_id = l.order_id
WHERE ABS(ROUND(o.order_total_amount - l.line_revenue_total, 2)) > 0.01
ORDER BY ABS(ROUND(o.order_total_amount - l.line_revenue_total, 2)) DESC, o.order_id
LIMIT 20

### Source Discovery Assertions

These read-only assertions fail fast if the training dataset no longer matches the course scenario. They protect the demo from silently drifting away from the expected table grain and status coverage.

Expected observation: the assertion summary returns `OK` for every check.

In [0]:
checks = []

orders = spark.sql(f"""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT order_id) AS distinct_keys
FROM {SILVER}.sales_orders
""").first().asDict()
checks.append(("sales_orders_order_id_unique", orders["rows"] == orders["distinct_keys"], str(orders)))

lines = spark.sql(f"""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_keys
FROM {SILVER}.order_lines
""").first().asDict()
checks.append(("order_lines_line_id_unique", lines["rows"] == lines["distinct_keys"], str(lines)))

status_count = spark.sql(f"""
SELECT COUNT(DISTINCT status) AS status_count
FROM {SILVER}.sales_orders
""").first()["status_count"]
checks.append(("multiple_status_values_present", status_count >= 3, str(status_count)))

risk_rows = spark.sql(f"""
SELECT SUM(affected_rows) AS affected_rows
FROM (
  SELECT COUNT(*) AS affected_rows
  FROM {SILVER}.sales_orders
  WHERE status NOT IN ('Completed', 'Returned', 'Cancelled')
  UNION ALL
  SELECT COUNT(*)
  FROM {SILVER}.sales_orders
  WHERE order_date > current_date()
  UNION ALL
  SELECT COUNT(*)
  FROM {SILVER}.order_lines
  WHERE unit_price IS NULL
  UNION ALL
  SELECT COUNT(*)
  FROM {SILVER}.order_lines l
  LEFT ANTI JOIN {SILVER}.customers c ON l.customer_id = c.customer_id
  UNION ALL
  SELECT COUNT(*)
  FROM {SILVER}.order_lines l
  LEFT ANTI JOIN {SILVER}.products p ON l.product_id = p.product_id
)
""").first()["affected_rows"]
checks.append(("source_risk_register_executed", risk_rows >= 0, str(risk_rows)))

failed = [name for name, ok, _ in checks if not ok]
if failed:
    raise Exception(f"Source discovery assertions failed: {failed}")

display(spark.createDataFrame([
    (name, observed, "OK") for name, _, observed in checks
], ["check_name", "observed_value", "status"]))

### Discovery Checklist

Before leaving Day1_01, participants should be able to answer these questions from query results, not from assumptions:

- What is the grain of each source table?
- Which status values exist, and which ones should count as revenue?
- Which keys will join customer and product dimensions?
- Which date drives reporting and refresh windows?
- Which fields are measures that require careful aggregation?

Expected observation: if any answer is unclear, do not start Gold modeling yet.

## Summary and Next Step

You have seen the platform foundation: Lakehouse, Delta tables, SQL Warehouse, cost guardrails and analyst tooling. The next notebook focuses on SQL programming patterns and analytics architecture, without building the final workshop model.